In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import joblib
import os

def visualizar_importancia_variables(proteinas, directory='./liposome_models-np_type', top_n=10, export_dir='./graficos_pdf'):
    """
    Carga los modelos de las proteínas, calcula la importancia global de cada variable,
    y grafica el Top N de los atributos más influyentes para cada proteína.
    """

    # Crear la carpeta para los PDFs si no existe
    os.makedirs(export_dir, exist_ok=True)
    
    # 1. Cargar los nombres de las características
    columns_filepath = os.path.join(directory, 'training_columns.joblib')
    try:
        feature_names = joblib.load(columns_filepath)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo en {columns_filepath}.")
        return
    
    # Configuración visual general
    sns.set_theme(style="whitegrid")
    
    for target in proteinas:
        filepath = os.path.join(directory, f'random_forest_regressor_model_{target}.joblib')
        
        try:
            modelo = joblib.load(filepath)
        except FileNotFoundError:
            print(f"Advertencia: Modelo para {target} no encontrado. Se omite.")
            continue
            
        # 2. Obtener la importancia de las variables del modelo
        importancias = modelo.feature_importances_
        
        # 3. Crear un DataFrame para ordenar y manejar los datos fácilmente
        df_importancias = pd.DataFrame({
            'Atributo': feature_names,
            'Importancia': importancias
        })
        
        # Ordenar de mayor a menor y tomar el Top N
        df_importancias = df_importancias.sort_values(by='Importancia', ascending=False).head(top_n)
        
        # 4. Graficar
        plt.figure(figsize=(10, 6))
        barplot = sns.barplot(x='Importancia', y='Atributo', data=df_importancias, palette='viridis')
        
        plt.title(f'Top {top_n} Variables más Importantes - Proteína {target}', fontsize=14, fontweight='bold')
        plt.xlabel('Importancia Relativa (Reducción del Error)', fontsize=12)
        plt.ylabel('Atributos del Liposoma', fontsize=12)
        
        # Añadir los valores numéricos al lado de cada barra
        for p in barplot.patches:
            width = p.get_width()
            plt.text(width + 0.005, p.get_y() + p.get_height()/2. + 0.1, '{:1.3f}'.format(width), ha="left")
            
        plt.tight_layout()

        # --- EXPORTAR A PDF ---
        pdf_path = os.path.join(export_dir, f'Importancia_Variables_{target}.pdf')
        plt.savefig(pdf_path, format='pdf', bbox_inches='tight')
        print(f"✅ Gráfico guardado en: {pdf_path}")

        plt.show()

# Lista de proteínas
proteinas_a_evaluar = ["P02768", "P01024", "P0C0L5", "P0C0L4", "P0D0X6", "P02741", "P11226"]

# Ejecutar mostrando el Top 10 de variables
visualizar_importancia_variables(proteinas_a_evaluar, top_n=10)